# 01. From four sources to one panel

**Owner:** Roger  \
**Inputs:** `data/processed/panel_municipality_year.parquet` and
`data/processed/fire_panel_municipality_year.parquet`, both built by `make data`  \
**Outputs:** figures to `reports/figures/`

## Purpose

Four raw sources — EFFIS burn perimeters, ICNF fire statistics, INE demography and
GADM boundaries — become one panel keyed on `(municipality, year)`. This chapter
explains how, and shows the evidence for the decisions that shape it.

**The cleaning itself does not happen here.** It lives in `wildfires.pipeline` and
`wildfires.merge`, runs as `make data`, and is covered by tests. Putting it in a
notebook would mean four chapters each cleaning slightly differently — the failure
this project is organised to avoid. What belongs here is the reasoning: which
records survive, which do not, and what a reader must know before trusting a
column.

Every number below is computed from the built artifacts, not typed in. If the
pipeline changes, this chapter's claims change with it or the notebook fails to run.

## Setup

In [ ]:
import json
import sys
import warnings
from pathlib import Path

# Make the src-layout package importable when the notebook kernel is not installed with -e .
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from wildfires.config import CONVENTIONS, PATHS
from wildfires.io import load_fire_panel, load_municipalities, load_panel
from wildfires.viz import apply_theme, save_figure

apply_theme()
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Two panels, not one

`make data` writes two analysis-ready tables. The reason is a hard constraint, not
a preference: INE's *Anuário Estatístico Regional* publishes municipality
demography for **2019–2024 only**. Joining demography onto fire history would
throw away the eighteen earlier years of fire records.

So the fire history is kept whole in its own panel, and the demographic join
produces a second, shorter one.

In [ ]:
panel = load_panel()            # fire x demography, 2019-2024
fire_panel = load_fire_panel()  # fire only, 2001-2025

summary = pd.DataFrame(
    {
        "rows": [len(fire_panel), len(panel)],
        "columns": [fire_panel.shape[1], panel.shape[1]],
        "first_year": [fire_panel.year.min(), panel.year.min()],
        "last_year": [fire_panel.year.max(), panel.year.max()],
        "municipalities": [fire_panel.dtcc.nunique(), panel.dtcc.nunique()],
    },
    index=["fire_panel_municipality_year", "panel_municipality_year"],
)
summary

The fire panel is **not** a clean 278 × 25 rectangle, and asserting that it should
be would be wrong. ICNF's municipality coverage climbs through the early 2000s and
never quite locks at 278: five later years report 277. The years below are every
one that departs from 278.

In [ ]:
coverage = fire_panel.groupby("year").dtcc.nunique().rename("municipalities")
incomplete = coverage[coverage != 278]

print(f"Years at the full 278: {(coverage == 278).sum()} of {len(coverage)}")
incomplete.to_frame().T

## Why 278 municipalities and not 308

Portugal has 308 municipalities (*concelhos*). The panel carries 278. Every step of
that reduction is a deliberate exclusion, not attrition.

In [ ]:
gadm = load_municipalities()
mainland = load_municipalities(mainland_only=True)

islands = gadm[gadm.dtcc.notna() & gadm.dtcc.str.startswith(("41", "42"))]

funnel = pd.DataFrame(
    [
        ("GADM level-2 municipalities", len(gadm), ""),
        ("carry a usable code", int(gadm.dtcc.notna().sum()),
         f"{int(gadm.dtcc.isna().sum())} Madeira/Azores rows have no code"),
        ("mainland only", len(mainland),
         f"{len(islands)} coded Azorean municipalities have no ICNF counterpart"),
        ("in the analysis panel", panel.dtcc.nunique(),
         "every mainland municipality matched INE"),
    ],
    columns=["step", "municipalities", "why"],
)
funnel

The last line is the one that matters. ICNF covers mainland Portugal and INE covers
308 municipalities; the 278 in both are the analysis population, and **not one of
them is lost in the join**. The panel is exactly 278 × 6 = 1,668 rows, which
`make data` refuses to emit if it is not.

### The trap: GADM writes "NA" as a string

GADM's `CC_2` code for the 23 Madeira and Azores municipalities is the literal text
`"NA"`, not a missing value. Read naively, all 23 share one key and a join collapses
them into a single row — silently, with no error.

In [ ]:
with PATHS["raw"]["gadm_municipal"].open(encoding="utf-8") as fh:
    raw_cc2 = pd.Series([f["properties"]["CC_2"] for f in json.load(fh)["features"]])

literal_na = raw_cc2[raw_cc2 == "NA"]
print(f'GADM rows whose raw CC_2 is the literal string "NA": {len(literal_na)}')
print(f"Distinct keys they collapse to in a naive join: {literal_na.nunique()}")
print(f"After load_municipalities: {int(gadm.dtcc.isna().sum())} genuinely missing")

There is a second reason the code is the only safe key: 308 municipalities share
only 306 distinct names.

In [ ]:
repeated = gadm.municipality.value_counts()
repeated[repeated > 1].rename("municipalities sharing the name").to_frame()

`Calheta` exists in both Madeira and the Azores; `Lagoa` in both the Algarve and the
Azores. Joining on name would merge them. **Join on `dtcc`, never on name.**

## The 2017 burned-area seam

This is the single most consequential thing to know about the panel.

ICNF publishes burned area under two conventions that answer different questions:

| Family | Source field | Measures |
|---|---|---|
| `burned_ha_*` | `AreaArd*_NoConcelho` | area that burned **within** this municipality |
| `burned_ha_*_ignited` | `AreaArd*_IncendioInicioConc` | area of fires that **ignited** in this municipality |

ICNF switched between them in 2017. The pipeline carries both, distinctly named, and
never coalesces them. Here is the coverage, counted from the fire panel.

In [ ]:
seam = fire_panel.groupby("year").agg(
    within=("burned_ha_total", "count"),
    ignited=("burned_ha_total_ignited", "count"),
)
both_present = fire_panel.burned_ha_total.notna() & fire_panel.burned_ha_total_ignited.notna()
seam["overlap"] = both_present.groupby(fire_panel.year).sum()
seam.T

Read the `overlap` row: it is zero in every year. The two conventions never coexist,
so **there is no year in which they can be calibrated against each other**.

In [ ]:
national = fire_panel.groupby("year")[["burned_ha_total", "burned_ha_total_ignited"]].sum(
    min_count=1
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(national.index, national.burned_ha_total_ignited / 1000,
        marker="o", linewidth=2, label="area of fires that ignited here (2001-2016)")
ax.plot(national.index, national.burned_ha_total / 1000,
        marker="o", linewidth=2, label="area burned within (2017-2025)")
ax.axvline(2016.5, color="0.4", linestyle="--", linewidth=1)
ax.annotate("convention changes\nno overlap year", xy=(2016.5, ax.get_ylim()[1] * 0.82),
            xytext=(8, 0), textcoords="offset points", fontsize=9, color="0.3")

ax.set_title("Two burned-area conventions meeting at 2017, national totals")
ax.set_xlabel("Year")
ax.set_ylabel("Thousand hectares")
ax.legend()
fig.tight_layout()
save_figure(fig, "icnf_burned_area_convention_seam")
plt.show()

The two lines abut; they do not overlap. Drawing them as one continuous series would
be a splice of two different measurements presented as a trend.

**What this means for analysis.** No single ICNF burned-area column spans 2017. Any
trend crossing that year must splice the two, explicitly and in one place, and must
be labelled a splice rather than a measurement — chapter 03 does exactly this for its
country-level series. Inside the analysis panel's 2019–2024 window the question does
not arise: only the `_NoConcelho` family has rows there, so **every `*_ignited`
column is 100% empty by construction**. They are kept rather than dropped because an
explicitly-empty documented column is safer than a column that quietly disappeared.

## What is missing, and why

Nothing in this project is imputed, interpolated or smoothed. Missing stays missing.
That makes the null rates meaningful, so they are worth reading carefully — the
groups below mean quite different things.

In [ ]:
nulls = panel.isna().mean().mul(100).round(2)


def group(column: str) -> str:
    if column.endswith("_ignited"):
        return "ICNF pre-2017 convention"
    if column.startswith(("effis_", "lc_", "percna2k")):
        return "EFFIS"
    if column.startswith("burned_ha") or column == "burn_rate":
        return "ICNF 2017+ convention"
    return "INE demography / keys"


by_group = (
    nulls.groupby(nulls.index.map(group))
    .agg(columns="size", min_null_pct="min", max_null_pct="max")
    .sort_values("max_null_pct", ascending=False)
)
by_group

Three different stories in one table:

- **ICNF pre-2017 convention, 100%.** Empty by construction, explained above.
- **EFFIS, ~59%.** Not a data failure — see below.
- **ICNF 2017+, 0.12%.** Two rows.
- **INE demography, 0%.** Every municipality-year has every demographic column.

The two missing ICNF rows are worth naming rather than waving at:

In [ ]:
panel.loc[panel.burned_ha_total.isna(), ["dtcc", "territory", "year", "n_fires"]]

Both recorded zero fires that year, so there was no area to report. A blank here
means "nothing burned", which is why the pipeline does not fill it with a zero it
did not observe.

### EFFIS nulls are not fire-free years

It is tempting to read a null EFFIS column as "no fire". That reading is wrong, and
it would bias any model that treats those rows as zeros.

In [ ]:
no_effis = panel[panel.effis_n_fires.isna()]

pd.Series(
    {
        "panel rows with no EFFIS record": len(no_effis),
        "…of which ICNF also recorded zero fires": int((no_effis.n_fires == 0).sum()),
        "…of which ICNF recorded one or more fires": int((no_effis.n_fires > 0).sum()),
        "median ICNF fire count in those rows": no_effis.n_fires.median(),
        "largest ICNF fire count in those rows": no_effis.n_fires.max(),
    },
    name="count",
).to_frame()

Almost every EFFIS-null row had fires — a median of 13, and one municipality-year
with 266. EFFIS maps burn *perimeters* from satellite imagery, so a fire too small
to be resolved leaves no polygon. A null therefore means **"no fire large enough to
be mapped"**, not "no fire".

Two consequences. First, EFFIS counts and ICNF counts measure different things and
will never agree; the gap between them is itself informative. Second, EFFIS lowered
its minimum mapped fire size around 2020, so its counts are not comparable across
that change without a constant size floor — `CONVENTIONS["min_fire_ha"]`, printed below, applied via
`wildfires.clean.apply_size_floor`.

EFFIS coverage also begins in 2016, well after ICNF's 2001.

### INE breaks in series

INE marks a break in series with `┴`. The pipeline records those flags as metadata
and **never alters the values** they mark — an adjusted number would be this
project's invention, not INE's measurement.

In [ ]:
pd.read_csv(PATHS["processed"]["series_breaks"])

The one that reaches the panel is population density at 2021, re-based on Censos
2021. Density is published for every municipality-year, so nothing looks missing —
which is exactly why the flag matters. **A 2020 → 2021 density change is not a
like-for-like comparison.**

## The contract

`make data` checks both panels against a contract before writing them and exits
non-zero if any check fails. A panel that multiplied rows in a merge, lost
municipalities in a join, or silently emptied a column would be caught here rather
than discovered as a strange coefficient three chapters later.

This is the report from the build that produced the files read above.

In [ ]:
from IPython.display import Markdown

report = PATHS["processed"]["validation_report"].read_text(encoding="utf-8")

# The per-column null-rate appendix runs to 69 rows; it stays in the file rather
# than in the chapter. Everything above it is the checks themselves.
checks_only = report.split("## Null rates")[0].rstrip()
print(f"Size floor in force: CONVENTIONS['min_fire_ha'] = {CONVENTIONS['min_fire_ha']} ha")
Markdown(checks_only)

## Takeaways

Three to five bullets. These get lifted verbatim into `05_discussion.ipynb`, so
write them as claims, not as descriptions of what you did.

-
-
-